# 已保存分钟响应的右端标签资格
固定两日、两symbol；不搜索偏移，不访问网络。按 plan.json 验证完整网格和 OHLCV 聚合，再做隔离 Qlib 读回。

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/"AGENTS.md").exists(): root=root.parent
sys.path.insert(0,str(root))
from src.data.diagnostic_forward_bar_mapping import run
result=run(root)
result

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'diagnostic_forward_bar_mapping_20261016',
 'plan_sha256': '00b8b8a366046e9765e2c0950cccc255af78b27e66a486d285d3b00c9339ffcf',
 'passed': False,
 'failure': {'type': 'RecorderInitializationError',
  'detail': "Please don't reinitialize Qlib if QlibRecorder is already activated. Otherwise, the experiment stored location will be modified."},
 'comparison': [{'symbol': 'SOXL',
   'windows': 26,
   'passed': True,
   'max_absolute_difference': {'open': 0.0,
    'high': 0.0,
    'low': 0.0,
    'close': 0.0,
    'volume': 0.0}},
  {'symbol': 'SOXX',
   'windows': 26,
   'passed': True,
   'max_absolute_difference': {'open': 0.0,
    'high': 0.0,
    'low': 0.0,
    'close': 0.0,
    'volume': 0.0}}],
 'network_attempts': 0,
 'model_calls': 0,
 'fits': 0,
 'strategy_replays': 0,
 'forward_days': 0,
 'pilot_ready': False,
 'qualified_gate': None,
 'rows': None,
 'recorder_id': '0b98c0e31bc5468594d59e66214527df',
 'experiment_id': '364729424211328971',
 'recorder_local_dir': '/U

In [2]:
import json,pickle,os
from hashlib import sha256
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.qlib_native import build_qlib_store_from_frame
from src.data.diagnostic_forward_inputs import network_denied
from src.data.minute_intake import atomic_json
card=root/'research/diagnostic_forward_bar_mapping_20261016'
original=json.loads((card/'summary.json').read_text())
prior_manifest=json.loads((card/'verification_manifest.json').read_text())
assert original['failure']['type']=='RecorderInitializationError'
assert all(x['passed'] for x in original['comparison'])
assert not (card/'native_repair_receipt.json').exists(),'no repair retry'
(card/'verification_manifest_before_native_repair.json').write_bytes((card/'verification_manifest.json').read_bytes())
artifacts=Path(original['recorder_local_dir'])/'artifacts'
for item in prior_manifest['entries']:assert sha256((card/item['path']).read_bytes()).hexdigest()==item['sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_TELEMETRY']='true';os.environ['DO_NOT_TRACK']='true'
fields=['open','high','low','close','volume'];parts=[]
for symbol in ['SOXL','SOXX']:
    frame=pickle.loads((artifacts/('new_'+symbol+'/original')).read_bytes())['payload']
    part=frame[fields].copy();part['datetime']=pd.to_datetime(frame.time_key);part['instrument']=symbol;parts.append(part)
source=pd.concat(parts,ignore_index=True);provider=root/'data/qlib/diagnostic_forward_bar_mapping_20261016'
with network_denied() as denied:
    manifest=build_qlib_store_from_frame(source,provider_uri=provider,freq='1min',region='us',as_of='2026-10-14',field_map={x:x for x in fields},source='Saved Futu NONE original end labels; bounded mapping only, no live availability/finality qualification')
    qlib.init(provider_uri=str(provider),region='us',kernels=1)
    expected=pd.date_range('2026-10-14 09:31:00','2026-10-14 16:00:00',freq='min',name='datetime')
    calendar=pd.DatetimeIndex(D.calendar(start_time=expected[0],end_time=expected[-1],freq='1min'),name='datetime');pd.testing.assert_index_equal(calendar,expected)
    readback=D.features(['SOXL','SOXX'],['$'+x for x in fields],start_time=expected[0],end_time=expected[-1],freq='1min');quant=[]
    for symbol in ['SOXL','SOXX']:
        raw=source[source.instrument.eq(symbol)].set_index('datetime')[fields];got=readback.xs(symbol,level='instrument');pd.testing.assert_index_equal(got.index.rename('datetime'),raw.index)
        np.testing.assert_array_equal(got.to_numpy(float),raw.to_numpy(dtype=np.float32).astype(float));quant.append({'symbol':symbol,'rows':len(raw),'max_absolute_float32_quantization':float(np.abs(got.to_numpy(float)-raw.to_numpy(float)).max())})
    R.set_uri(str(card/'qlib_recorder'));rec=R.get_recorder(recorder_id=original['recorder_id'],experiment_id=original['experiment_id'])
    receipt={'decision':'ROOT_APPROVED_ONE_SOFTWARE_CONTINUATION','reason':'Move Qlib builder/init outside active R.start; reuse only saved new_SYMBOL/original; no recomparison/refetch','original_failure':original['failure'],'namespace':'native_repair_1','network_attempts':len(denied),'new_model_calls':0,'new_policy_replays':0}
    result={'passed':True,'rows':len(readback),'calendar_rows':len(calendar),'quantization':quant,'gate':'bounded_saved_completed_history_end_label_and_Qlib','pilot_ready':False,'forward_days':0}
    notebook=json.loads((card/'diagnostic_forward_bar_mapping_20261016.ipynb').read_text());cell=next(x for x in notebook['cells'] if x.get('id')=='native_repair_1');cell_source=''.join(cell['source'])
    rec.save_objects(artifact_path='native_repair_1',repair_source=cell_source,approval=receipt,source=source,readback=readback,calendar=calendar,provider_manifest=manifest,result=result)
    rec.log_metrics(native_repair_registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
for item in prior_manifest['entries']:assert sha256((card/item['path']).read_bytes()).hexdigest()==item['sha256'],'original artifact changed'
receipt['all_original_artifact_hashes_unchanged']=True;receipt['original_artifact_count']=len(prior_manifest['entries'])
atomic_json(receipt,card/'native_repair_receipt.json');atomic_json(manifest,card/'provider_manifest.json')
summary={**original,'passed':True,'failure':None,'original_failure':original['failure'],'original_summary_artifact_unchanged':True,'native_repair':result,'qualified_gate':result['gate'],'rows':780}
atomic_json(summary,card/'summary.json');atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary


2026/10/16 02:30:49 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[21700:MainThread](2026-10-16 02:30:49,904) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[21700:MainThread](2026-10-16 02:30:49,905) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[21700:MainThread](2026-10-16 02:30:49,906) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/diagnostic_forward_bar_mapping_20261016')}


[21700:MainThread](2026-10-16 02:30:49,911) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[21700:MainThread](2026-10-16 02:30:49,914) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/diagnostic_forward_bar_mapping_20261016')}


[21700:MainThread](2026-10-16 02:30:49,921) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[21700:MainThread](2026-10-16 02:30:49,921) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[21700:MainThread](2026-10-16 02:30:49,921) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/diagnostic_forward_bar_mapping_20261016')}


{'experiment': 'diagnostic_forward_bar_mapping_20261016',
 'plan_sha256': '00b8b8a366046e9765e2c0950cccc255af78b27e66a486d285d3b00c9339ffcf',
 'passed': True,
 'failure': None,
 'comparison': [{'symbol': 'SOXL',
   'windows': 26,
   'passed': True,
   'max_absolute_difference': {'open': 0.0,
    'high': 0.0,
    'low': 0.0,
    'close': 0.0,
    'volume': 0.0}},
  {'symbol': 'SOXX',
   'windows': 26,
   'passed': True,
   'max_absolute_difference': {'open': 0.0,
    'high': 0.0,
    'low': 0.0,
    'close': 0.0,
    'volume': 0.0}}],
 'network_attempts': 0,
 'model_calls': 0,
 'fits': 0,
 'strategy_replays': 0,
 'forward_days': 0,
 'pilot_ready': False,
 'qualified_gate': 'bounded_saved_completed_history_end_label_and_Qlib',
 'rows': 780,
 'recorder_id': '0b98c0e31bc5468594d59e66214527df',
 'experiment_id': '364729424211328971',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/diagnostic_forward_bar_mapping_20261016/qlib_recorder/364729424211328971/0b98c0e31bc5468594d59e66

# 固定已保存响应的端点映射结论

固定比较通过：2026-07-29 SOXL/SOXX 各 26 个完整 15 分钟窗口，按 (end−15min,end] 聚合 open/high/low/close/volume，52×5 个值全部精确相等（rtol=0、atol=1e−7）。未内连接删除缺失窗口，未搜索偏移或日期。

2026-10-14 两份已保存 NONE 响应共 780 行，原 time_key 直接作为声明 interval_end，interval_start=end−1min；未额外 +1min。隔离 Qlib 的 390 个日历端点、780 行 OHLCV 与原数组 float32 转换结果逐值相等。SOXL/SOXX 原值至 float32 最大绝对量化分别为 3.8086e−6、3.02734e−5。

最初固定比较已成功，但在 active Recorder 中初始化新 Qlib provider 触发 RecorderInitializationError。获准的一次软件续接在 Recorder 上下文外只使用已保存 payload 完成读回，并追加 native_repair_1 namespace。原 22 个 artifact hash 全部不变，原失败 summary、workflow/source 与 manifest 单独保留。旧 registered_artifact_count=1 是原实现固定值；新 native_repair_registered_artifact_count 记录实际对象数。

证据边界：old K1M 的 QFQ 有原 manifest 证明；direct K15M 只有 path/hash，其 autype、SDK、receipt 未认证。因此这是同日保存价量的经验聚合一致，不是已认证同调整接口，也不是旧 QFQ 与新 NONE 价格等价。固定比较支持声明的已完成历史分钟 end-label 映射，不证明所有日期/vendor 语义、live prefix 可用时间或不可修订 finality。

本次只解除固定已保存 completed-history end-label/Qlib 读回门。原 start-grid 探测失败不回填，未修改原三执行模块；0 网络、0 模型/fit/replay、0 前向天数。daily collector 角色映射、实际未来 receipt→seal→later quote 会话及 pilot 仍未完成，不可据此宣称策略达标或直接启动。

下一步：把已审定 end-label 合同以新代码身份接入前向适配器，明确剩余 daily source-role gate 后再独立审定完整会话激活；不重新采集本轮数据。

独立评估接受本限定结果；负责人从NAS空目录恢复核对29个对象，全部哈希一致。源响应和原失败对象均保持原身份。
